In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Pusa_Delhi_IMD_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,315.0,151.0,NaN,120.0,196.0,189.0,NaN,85.0,NaN,119.0,338.0,263.0
1,2,315.0,127.0,NaN,137.0,262.0,133.0,73.0,55.0,NaN,123.0,307.0,265.0
2,3,NaN,NaN,NaN,NaN,313.0,137.0,66.0,57.0,59.0,113.0,362.0,235.0
3,4,335.0,291.0,NaN,170.0,NaN,187.0,61.0,41.0,52.0,215.0,362.0,131.0
4,5,322.0,134.0,NaN,186.0,322.0,216.0,68.0,36.0,59.0,NaN,347.0,112.0
5,6,NaN,118.0,NaN,193.0,291.0,154.0,64.0,NaN,53.0,102.0,320.0,171.0
6,7,265.0,NaN,NaN,198.0,301.0,235.0,54.0,NaN,57.0,92.0,368.0,205.0
7,8,307.0,NaN,NaN,220.0,248.0,216.0,56.0,52.0,65.0,122.0,NaN,NaN
8,9,292.0,NaN,NaN,209.0,240.0,138.0,67.0,52.0,84.0,111.0,340.0,157.0
9,10,191.0,NaN,171.0,NaN,227.0,147.0,109.0,57.0,87.0,88.0,324.0,186.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,315.000000,108.363636,161.0,167.862069,179.058824,189.000000,73.176471,85.00000,71.878788,119.000,338.0000,263.00
1,2,315.000000,108.363636,161.0,137.000000,179.058824,133.000000,73.000000,55.00000,71.878788,123.000,307.0000,265.00
2,3,246.481481,108.363636,161.0,167.862069,179.058824,137.000000,66.000000,57.00000,59.000000,113.000,362.0000,235.00
3,4,335.000000,108.363636,161.0,170.000000,179.058824,187.000000,61.000000,41.00000,52.000000,215.000,362.0000,131.00
4,5,322.000000,108.363636,161.0,186.000000,179.058824,216.000000,68.000000,36.00000,59.000000,167.875,347.0000,112.00
5,6,246.481481,108.363636,161.0,193.000000,179.058824,154.000000,64.000000,66.40625,53.000000,102.000,320.0000,171.00
6,7,265.000000,108.363636,161.0,198.000000,179.058824,132.151515,54.000000,66.40625,57.000000,92.000,368.0000,205.00
7,8,307.000000,108.363636,161.0,220.000000,179.058824,216.000000,56.000000,52.00000,65.000000,122.000,304.0625,205.36
8,9,292.000000,108.363636,161.0,209.000000,179.058824,138.000000,67.000000,52.00000,84.000000,111.000,340.0000,157.00
9,10,191.000000,108.363636,161.0,167.862069,179.058824,147.000000,109.000000,57.00000,87.000000,88.000,324.0000,186.00
